# Train fall detection models on Colab (Lin et al. 2021)

1. On your machine: `bash scripts/make_colab_bundle.sh` (add `WITH_VIDEOS=1` to re-extract poses here on the GPU).
2. Upload `falldet_colab.zip` to Google Drive as **MyDrive/falldet/falldet_colab.zip**.
3. In Colab: *Runtime → Change runtime type → GPU*, then run the cells top to bottom.

Runs are written to **MyDrive/falldet/runs**, so they survive a disconnect. Download that folder into
`runs/` of the local project to use them with `falldet.evaluate`, `falldet.demo` or `falldet.summarize`.

**In VS Code:** Select Kernel → Colab → New Colab Server with a T4 GPU, then
Command Palette → "Colab: Mount Google Drive to Server...", then **Run All**.

In [5]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [6]:
import os

DRIVE = "/content/drive/MyDrive/falldet"
BUNDLE = f"{DRIVE}/falldet_colab.zip"
RUNS = f"{DRIVE}/runs"
WORK = "/content/falldet"

# In VS Code, mount Drive with the Command Palette command "Colab: Mount Google Drive to Server..."
# if the call below does not open a sign-in prompt. In the browser, the call below is enough.
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
assert os.path.isfile(BUNDLE), f"Upload falldet_colab.zip to Google Drive as {BUNDLE}"
print("bundle found:", BUNDLE)


Mounted at /content/drive
bundle found: /content/drive/MyDrive/falldet/falldet_colab.zip


In [7]:
!rm -rf {WORK} && mkdir -p {WORK} {RUNS} && unzip -q {BUNDLE} -d {WORK}
%cd {WORK}
# Colab already ships a CUDA build of torch/torchvision; this only adds what is missing.
!pip install -q -e .
!python -c "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"

/content/falldet
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 10.0 MB/s eta 0:00:00
  Building editable for falldet (pyproject.toml) ... done
torch 2.11.0+cu128 | cuda True


## Optional: re-extract poses on the GPU with a larger model
Needs a bundle made with `WITH_VIDEOS=1`. The local poses were made with `yolo11n-pose` on CPU;
`yolo11m-pose` is more accurate. Train on them with `data.real_dir=data/poses/urfd_m`.

In [8]:
import os
if os.path.isdir("data/urfd/videos"):
    !python -m falldet.extract_poses --videos data/urfd/videos --segments data/urfd/segments.csv \
        --out data/poses/urfd_m --weights yolo11m-pose.pt --resize-width 0 --device 0
    !python -m falldet.pose_quality data/poses/urfd data/poses/urfd_m
else:
    print("No videos in the bundle (made without WITH_VIDEOS=1): skipping pose re-extraction.")

No videos in the bundle (made without WITH_VIDEOS=1): skipping pose re-extraction.


## Main models (paper setting: 512 hidden nodes, RP-normalization, lr 0.01, 500 epochs, full batch)

In [9]:
for model in ["lstm", "gru", "rnn"]:
    !python -m falldet.train --out {RUNS}/urfd/{model} model.name={model} train.device=cuda | tail -n 14
# the paper's plain random split, to measure how much window leakage inflates the score
!python -m falldet.train --out {RUNS}/urfd/lstm_paper_split data.group_split=false train.device=cuda | tail -n 14

epoch 500  loss 0.0001  train_acc 1.000  (0.2s)

TEST (best checkpoint)
accuracy  91.67%   precision  93.33%   recall  91.67%   F1  91.85%
sensitivity  87.50%   specificity 100.00%

class               prec  recall      F1     n
fall               1.000   0.875   0.933    16
no_fall            0.800   1.000   0.889     8

confusion matrix (rows = true, cols = predicted)
            0     1
fall       14     2
no_fall     0     8
epoch 500  loss 0.0000  train_acc 1.000  (0.2s)

TEST (best checkpoint)
accuracy  95.83%   precision  96.30%   recall  95.83%   F1  95.89%
sensitivity  93.75%   specificity 100.00%

class               prec  recall      F1     n
fall               1.000   0.938   0.968    16
no_fall            0.889   1.000   0.941     8

confusion matrix (rows = true, cols = predicted)
            0     1
fall       15     1
no_fall     0     8
epoch 500  loss 0.8140  train_acc 0.472  (0.2s)

TEST (best checkpoint)
accuracy  66.67%   precision  44.44%   recall  66.67%   F1  53

In [10]:
!python -m falldet.summarize {RUNS}/urfd

| run | model | norm | hidden | lr | accuracy | sensitivity | specificity | f1 | n_test |
|---|---|---|---|---|---|---|---|---|---|
| gru | gru | rp | 512 | 0.01 | 95.8 | 93.8 | 100.0 | 95.9 | 24 |
| lstm | lstm | rp | 512 | 0.01 | 91.7 | 87.5 | 100.0 | 91.9 | 24 |
| lstm_paper_split | lstm | rp | 512 | 0.01 | 81.5 | 61.5 | 100.0 | 80.7 | 27 |
| rnn | rnn | rp | 512 | 0.01 | 66.7 | 100.0 | 0.0 | 53.3 | 24 |


## All experiments of the paper (Tables 5-10): 39 runs
Finished runs are skipped, so the cell can be re-run after a disconnect.

In [11]:
RUN_FULL_EXPERIMENTS = True  # 39 runs, the longest part; set False to stop after the main models
if RUN_FULL_EXPERIMENTS:
    !OUT={RUNS}/lin2021 bash scripts/lin2021_experiments.sh
    !python -m falldet.summarize {RUNS}/lin2021 --csv {RUNS}/lin2021_results.csv

== norm_none_rnn
            0     1
fall       13     3
no_fall     3     5
== norm_minmax_rnn
            0     1
fall       16     0
no_fall     8     0
== norm_rp_rnn
            0     1
fall       16     0
no_fall     8     0
== norm_rp_interp_rnn
            0     1
fall        0    16
no_fall     0     8
== hidden_64_rnn
            0     1
fall       13     3
no_fall     0     8
== hidden_128_rnn
            0     1
fall       12     4
no_fall     0     8
== hidden_256_rnn
            0     1
fall       13     3
no_fall     1     7
== hidden_512_rnn
            0     1
fall       16     0
no_fall     8     0
== hidden_1024_rnn
            0     1
fall        0    16
no_fall     0     8
== lr_0.1_rnn
            0     1
fall       16     0
no_fall     5     3
== lr_0.01_rnn
            0     1
fall       16     0
no_fall     8     0
== lr_0.001_rnn
            0     1
fall       13     3
no_fall     0     8
== paper_split_rnn
            0     1
fall       13     0
no_fall    14